# ROS 2 playground (library / notes)

Scratchpad for learning `rclpy` against the running simulation before the logic moves into the
C++ nodes in `gambit_ros/`. Three things the sim pipeline needs from ROS:

1. **subscribe** to `/camera/image` (sensor_msgs/Image) and turn it into a numpy array,
2. **call** `/world/gambit_chess/set_pose` (ros_gz_interfaces/srv/SetEntityPose) to teleport,
3. **send** a `FollowJointTrajectory` goal to `/arm_controller/follow_joint_trajectory`.

Everything ROS is behind `try: import rclpy` so this notebook also runs without ROS; the
self-check only tests the pure helpers (message <-> numpy/dict conversion).

Start the sim first: `ros2 launch gambit_bringup gambit.launch.py` (or docs/gazebo.md by hand).

**TODO checklist**
- [x] Image message -> numpy without cv_bridge (rgb8/bgr8/mono8)
- [x] SetEntityPose / FollowJointTrajectory goals as plain dicts (and to ROS messages)
- [ ] `CameraGrabber` node: latest frame, wait-for-fresh-frame after a teleport
- [ ] `teleport()` service client (move into teleport.Ros2Backend when it works)
- [ ] `send_arm_goal()` action client with feedback/result handling
- [ ] measure: set_pose -> first image showing the change (latency), for capture.settle_s

In [ ]:
%run ./teleport.ipynb

In [ ]:
import numpy as np

try:
    import rclpy
    from rclpy.node import Node
    HAVE_ROS = True
except ImportError:          # no ROS here: helpers below still work
    rclpy, Node, HAVE_ROS = None, object, False
print("rclpy available:", HAVE_ROS)

ARM_JOINTS = ["shoulder_pan", "shoulder_lift", "elbow_flex", "wrist_flex", "wrist_roll"]
PARK = BOARD["arm"]["park"]

## Images

`sensor_msgs/Image` has `height, width, encoding, step, data`. `cv_bridge.CvBridge().imgmsg_to_cv2(msg,
"rgb8")` is the usual way; the pure-numpy version below avoids the cv_bridge/numpy version issues
that often appear on fresh installs. Gazebo publishes `rgb8` (R8G8B8 in the camera SDF).

In [ ]:
def image_msg_to_numpy(msg):
    """sensor_msgs/Image (or any object with the same fields) -> (H, W, 3) RGB or (H, W) uint8."""
    ch = {"rgb8": 3, "bgr8": 3, "mono8": 1}.get(msg.encoding)
    if ch is None:
        raise ValueError(f"unsupported encoding {msg.encoding}")
    buf = np.frombuffer(bytes(msg.data), dtype=np.uint8).reshape(msg.height, msg.step)
    img = buf[:, : msg.width * ch].reshape(msg.height, msg.width, ch)
    if msg.encoding == "bgr8":
        img = img[..., ::-1]
    return img[..., 0].copy() if ch == 1 else img.copy()

## Requests as dicts

Building requests as dicts keeps them testable and printable; `to_ros_*` convert at the last moment.

In [ ]:
def trajectory_goal(positions, duration_s=3.0, joints=ARM_JOINTS):
    """control_msgs/action/FollowJointTrajectory goal (one waypoint) as a dict."""
    assert len(positions) == len(joints)
    sec = int(duration_s)
    return {"trajectory": {"joint_names": list(joints),
                           "points": [{"positions": [float(p) for p in positions],
                                       "time_from_start": {"sec": sec, "nanosec": int(round((duration_s - sec) * 1e9))}}]}}

def to_ros_set_pose(req):
    """SetPose -> ros_gz_interfaces.srv.SetEntityPose.Request (needs ROS)."""
    from ros_gz_interfaces.srv import SetEntityPose
    from ros_gz_interfaces.msg import Entity
    r = SetEntityPose.Request()
    r.entity.name, r.entity.type = req.name, Entity.MODEL
    (r.pose.position.x, r.pose.position.y, r.pose.position.z) = req.position
    (r.pose.orientation.x, r.pose.orientation.y, r.pose.orientation.z, r.pose.orientation.w) = req.orientation
    return r

def to_ros_trajectory_goal(goal):
    """dict from trajectory_goal() -> FollowJointTrajectory.Goal (needs ROS)."""
    from control_msgs.action import FollowJointTrajectory
    from trajectory_msgs.msg import JointTrajectoryPoint
    from builtin_interfaces.msg import Duration
    g = FollowJointTrajectory.Goal()
    g.trajectory.joint_names = goal["trajectory"]["joint_names"]
    for p in goal["trajectory"]["points"]:
        pt = JointTrajectoryPoint(positions=p["positions"])
        pt.time_from_start = Duration(sec=p["time_from_start"]["sec"], nanosec=p["time_from_start"]["nanosec"])
        g.trajectory.points.append(pt)
    return g

## Nodes (TODO, need ROS)

Remember `use_sim_time: True` so timestamps follow Gazebo's /clock. In a notebook, spin with
`rclpy.spin_once(node, timeout_sec=0.1)` in a loop instead of `rclpy.spin` (which blocks forever).

In [ ]:
class CameraGrabber(Node):
    """TODO: keep the latest /camera/image as numpy.

    __init__: super().__init__('camera_grabber'); set use_sim_time; self.create_subscription(
    Image, '/camera/image', self._cb, qos_profile_sensor_data). _cb stores image_msg_to_numpy(msg)
    and msg.header.stamp. grab_after(stamp): spin_once until a frame newer than `stamp` arrives.
    """
    def __init__(self):
        if not HAVE_ROS:
            raise RuntimeError("needs rclpy")
        raise NotImplementedError("TODO: CameraGrabber")

def teleport(node, req, world=WORLD_NAME, timeout_s=2.0):
    """TODO: cli = node.create_client(SetEntityPose, f'/world/{world}/set_pose');
    cli.wait_for_service(timeout_s); fut = cli.call_async(to_ros_set_pose(req));
    rclpy.spin_until_future_complete(node, fut, timeout_sec=timeout_s); return fut.result().success"""
    raise NotImplementedError("TODO: teleport")

def send_arm_goal(node, goal, action="/arm_controller/follow_joint_trajectory"):
    """TODO: rclpy.action.ActionClient(node, FollowJointTrajectory, action); wait_for_server;
    send_goal_async(to_ros_trajectory_goal(goal), feedback_callback=...); wait for acceptance, then
    get_result_async(); check result.error_code == 0 (SUCCESSFUL)."""
    raise NotImplementedError("TODO: send_arm_goal")

## Try it (only with ROS + the sim running; edit and run by hand)

```python
rclpy.init()
node = rclpy.create_node('playground')
teleport(node, SetPose('w_P_5', square_center('e4')))
send_arm_goal(node, trajectory_goal([0.0, 0.2, 0.3, 1.4, 0.0]))
send_arm_goal(node, trajectory_goal(PARK))
node.destroy_node(); rclpy.shutdown()
```

## Self-check (no ROS needed)

In [ ]:
from types import SimpleNamespace

h, w = 2, 3
rgb = np.arange(h * w * 3, dtype=np.uint8).reshape(h, w, 3)
padded = np.zeros((h, w * 3 + 2), np.uint8); padded[:, : w * 3] = rgb.reshape(h, -1)   # step > width*3
msg = SimpleNamespace(height=h, width=w, encoding="rgb8", step=w * 3 + 2, data=padded.tobytes())
assert np.array_equal(image_msg_to_numpy(msg), rgb)
msg.encoding = "bgr8"
assert np.array_equal(image_msg_to_numpy(msg), rgb[..., ::-1])

g = trajectory_goal([0.0, 0.2, 0.3, 1.4, 0.0], duration_s=2.5)
assert g["trajectory"]["joint_names"] == ARM_JOINTS
assert g["trajectory"]["points"][0]["time_from_start"] == {"sec": 2, "nanosec": 500000000}
assert len(PARK) == 5
print("ros_playground self-check OK")